In [ ]:
import arcpy

# check out Spatial Analyst license
arcpy.CheckOutExtension("Spatial")

In [ ]:
# ── PATHS ────────────────────────────────────────────────────────────────────

# Input USC rasters (full high-res coverage)
usc_bathy = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\thesis_custom_rasters.gdb\USC10m_bathy_thesis"
usc_slope = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\thesis_custom_rasters.gdb\USC10m_slope_thesis"

# Input GBC rasters * with USC overlap area removed * (from script 2)
gbc_bathy_nooverlap = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\thesis_custom_rasters.gdb\GBC15a_bathy_thesis_NOOVERLAP"
gbc_slope_nooverlap = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\thesis_custom_rasters.gdb\GBC15a_slope_thesis_NOOVERLAP"

# Output GDB (write all results to thesis_custom_rasters.gdb, NOT to source GDBs)
out_gdb = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\thesis_custom_rasters.gdb"

#༝ Name output merged rasters (FINAL RASTERS THAT WILL BE USED IN ALL THESIS MAPS!)
bathy_merged_name = "MERGED_BATHY_THESIS"
slope_merged_name = "MERGED_SLOPE_THESIS"

In [ ]:
# ── SETTINGS ─────────────────────────────────────────────────────────────────
arcpy.env.overwriteOutput = True        # overwrite existing outputs on rerun
arcpy.env.compression = "NONE"
arcpy.env.workspace = out_gdb
arcpy.env.resamplingMethod = "BILINEAR" #✦ use bilinear interpolation to resample GBC rasters up to match 10-m res in USC raster

In [ ]:
# ── INSPECT INPUT RASTER PROPERTIES ──────────────────────────────────────────

# List input rasters to inspect paired with a readable label
inputs = [
    ("USC10m_bathy_thesis", usc_bathy),
    ("USC10m_slope_thesis", usc_slope),
    ("GBC15a_bathy_thesis_NOOVERLAP", gbc_bathy_nooverlap),
    ("GBC15a_slope_thesis_NOOVERLAP", gbc_slope_nooverlap),
]

#❋ Loop through each raster and print its key properties
for label, path in inputs:
    # wrap path in Raster object to access its properties
    r = arcpy.Raster(path)
    # header for current raster
    print(f"=== {label} ===")
    #༝ pixel type code 
    #ㅱ(F32 = 32-bit float, S16 = 16-bit signed int, etc.)
    print(f"  pixel type:   {r.pixelType}")
    #༝ cell size (reported separately for x and y); 
    #ㅱ expect ~10 for USC and coarse for GBC
    print(f"  cell size x:  {r.meanCellWidth}")
    print(f"  cell size y:  {r.meanCellHeight}")
    #༝ number of bands (expect 1 for single-band bathy/slope)
    print(f"  band count:   {r.bandCount}")
    #༝ hasRAT flags categorical rasters that carry a colormap mode; continuous bathy/slope should return False (no colormap, mode is inert)
    print(f"  has RAT:      {r.hasRAT}   (True = categorical, colormap mode may matter)")
    #༝ integer vs float: another continuous/categorical check
    print(f"  is integer:   {r.isInteger}")
    print()

In [ ]:
# ── HELPER FUNCTION ──────────────────────────────────────────────────────────
def mosaic(usc_raster, gbc_raster, out_name, label=""):

    #༝ pull spatial reference system used in USC raster (NAD_1983_UTM_Zone_20N)
    usc_crs = arcpy.Describe(usc_raster).spatialReference
    
    # print function report summary
    print(f"  Mosaicking: {label} → {out_name}")

    #❥ Run 'Mosaic To New Raster' tool (Data Management)
    arcpy.management.MosaicToNewRaster(

        # input rasters: USC10m_*_thesis + GBC15a_*_thesis_NOOVERLAP (* = bathy or slope)
        input_rasters=[usc_raster, gbc_raster], #✦ USC raster listed FIRST
        # output merged raster to ~\Bathymetry_geodatabases\thesis_custom_rasters.gdb
        output_location=out_gdb,
        # name output raster as "MERGED_*_THESIS" (* = bathy or slope)
        raster_dataset_name_with_extension=out_name,
        
        #༝ apply spatial reference system used in USC raster (NAD_1983_UTM_Zone_20N) to new merged raster
        coordinate_system_for_the_raster=usc_crs,

        #❋ set to match WIDEST pixel type across input rasters (32 bit float = pixel type will be a 32-bit data type supporting decimals)
        pixel_type="32_BIT_FLOAT", #ㅱ check
        #✦ force output to 10-m grid (upsample GBC raster to match USC raster native resolution) using BILINEAR INTERPOLATION set in arcpy.env.resamplingMethod = "BILINEAR" line above
        cellsize=10, #ㅱ check
        #❋ single-band raster for bathy or slope (not multiband imagery)
        number_of_bands=1, #ㅱ check
        #✦ "FIRST" means overlapping cells in merged rasters take value from the FIRST-LISTED raster in input_rasters=[] line above (usc_raster listed first = overlapping cells use only USC raster values, no GBC values)
        mosaic_method="FIRST",
        #ㅱ colormap not present???
        mosaic_colormap_mode="FIRST" #ㅱ check
    )

    print(f"  Saved: {out_gdb}\\{out_name}")

In [ ]:
# ── 1. MOSAIC: BATHYMETRY ────────────────────────────────────────────────────
print("=== Mosaic: Bathymetry ===")
mosaic(
    usc_raster = usc_bathy,
    gbc_raster = gbc_bathy_nooverlap,
    out_name   = bathy_merged_name,
    label      = "USC bathy + GBC (no overlap) bathy → merged"
)

In [ ]:
# ── 2. MOSAIC: SLOPE ─────────────────────────────────────────────────────────
print("=== Mosaic: Slope ===")
mosaic(
    usc_raster = usc_slope,
    gbc_raster = gbc_slope_nooverlap,
    out_name   = slope_merged_name,
    label      = "USC slope + GBC (no overlap) slope → merged"
)

In [ ]:
# ── ADD MERGED RASTERS TO CURRENT MAP ────────────────────────────────────────
print("=== Adding layers to map ===")

#⊹ should be working in current map (map_setup_thesis)
aprx = arcpy.mp.ArcGISProject("CURRENT")
m    = aprx.activeMap

#༝ merged output rasters
for out_name in [
    bathy_merged_name,
    slope_merged_name
]:
    m.addDataFromPath(out_gdb + "\\" + out_name)
    print(f"  Added output: {out_name}")

# release Spatial Analyst license
arcpy.CheckInExtension("Spatial")

print("=== Done ===")